In [ ]:
!pip install pyvis spacy stanza networkx
!python -m spacy download fr_core_news_lg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 34.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 MB 2.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
import sys
import os
PROJECT_PATH = "/content/drive/MyDrive/M1/AMS Projet 1/FinalProject"
sys.path.append(PROJECT_PATH)

In [ ]:
import importlib
# importlib.reload(importlib.import_module("nlp_extract_characters"))
# importlib.reload(importlib.import_module("nlp_utils"))
# importlib.reload(importlib.import_module("nlp_aliases"))
importlib.reload(importlib.import_module("nlp_cooccurrence"))
# importlib.reload(importlib.import_module("nlp_graph"))
# importlib.reload(importlib.import_module("nlp_visualize_web"))
importlib.reload(importlib.import_module("nlp_create_submission"))
importlib.reload(importlib.import_module("nlp_multi_ner"))
from nlp_multi_ner import ensemble_entities
from nlp_create_submission import process_chapter, generate_submission, generate_submission_parallel
from nlp_visualize_web import create_interactive_graph
from nlp_graph import generate_graph, save_graphml, visualize_graph
from nlp_cooccurrence import detect_cooccurrences
from nlp_extract_characters import is_valid_entity, extract_entities, count_entities, filter_persons, filter_locations
from nlp_utils import read_file, load_anti_dict
from nlp_aliases import group_aliases, alias_dictionary, merge_alias_counts

import networkx as nx

In [ ]:
# %%
# =============================================================================
# CONFIGURATION - ADJUST THESE PATHS
# =============================================================================

books_folder = "/content/drive/MyDrive/M1/AMS Projet 1/FinalProject/data"
books_config = [
    # (chapter_range, book_code, folder_path)
    (list(range(0, 19)), "paf", f"{books_folder}/prelude_a_fondation"),      # paf0 to paf18
    (list(range(0, 18)), "lca", f"{books_folder}/les_cavernes_d_acier"),     # lca0 to lca17
]

anti_dict_file = "antidict.txt"
output_csv = "submission.csv"
distance_max = 100  # Co-occurrence window size

# %%
# =============================================================================
# RUN SUBMISSION GENERATION - PARALLEL (FASTER!)
# =============================================================================

import time

print("🚀 Starting PARALLEL submission generation...")
print(f"Distance threshold: {distance_max} words")

start_time = time.time()

df_submission = generate_submission_parallel(
    books_config=books_config,
    anti_dict_file=anti_dict_file,
    output_csv=output_csv,
    distance_max=distance_max,
    n_processes=None  # Auto-detect CPUs (or set manually, e.g., n_processes=4)
)

elapsed_time = time.time() - start_time
print(f"\n⏱️  Total time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")

# %%
# =============================================================================
# ORIGINAL (SEQUENTIAL) VERSION - UNCOMMENT TO USE
# =============================================================================
# print("🚀 Starting submission generation...")
# print(f"Distance threshold: {distance_max} words")
# 
# df_submission = generate_submission(
#     books_config=books_config,
#     anti_dict_file=anti_dict_file,
#     output_csv=output_csv,
#     distance_max=distance_max
# )

# %%
# =============================================================================
# TEST MULTIPLE WINDOW SIZES IN PARALLEL (RECOMMENDED FOR OPTIMIZATION)
# =============================================================================

def test_window_sizes_parallel(window_sizes, books_config, anti_dict_file):
    """Test multiple window sizes and compare results"""
    import time
    import pandas as pd
    
    results = {}
    
    for distance in window_sizes:
        print(f"\n{'='*60}")
        print(f"🔍 Testing window size: {distance} words")
        print(f"{'='*60}")
        
        start_time = time.time()
        
        output_csv = f"submission_{distance}.csv"
        df = generate_submission_parallel(
            books_config=books_config,
            anti_dict_file=anti_dict_file,
            output_csv=output_csv,
            distance_max=distance,
            n_processes=None  # Auto-detect
        )
        
        elapsed = time.time() - start_time
        
        # Calculate statistics
        import io
        avg_nodes = df.apply(lambda row: nx.read_graphml(io.StringIO(row['graphml'])).number_of_nodes(), axis=1).mean()
        avg_edges = df.apply(lambda row: nx.read_graphml(io.StringIO(row['graphml'])).number_of_edges(), axis=1).mean()
        
        results[distance] = {
            'avg_nodes': round(avg_nodes, 2),
            'avg_edges': round(avg_edges, 2),
            'time_seconds': round(elapsed, 2)
        }
        
        print(f"   ⏱️  Completed in {elapsed:.2f}s")
        print(f"   📊 Avg nodes: {avg_nodes:.1f}, Avg edges: {avg_edges:.1f}")
    
    # Summary table
    print(f"\n{'='*60}")
    print("📊 WINDOW SIZE COMPARISON")
    print(f"{'='*60}")
    results_df = pd.DataFrame(results).T
    results_df.index.name = 'window_size'
    print(results_df)
    
    return results_df

# UNCOMMENT TO RUN WINDOW SIZE TESTS:
# window_sizes_to_test = [25, 50, 75, 100, 150]
# results_comparison = test_window_sizes_parallel(
#     window_sizes=window_sizes_to_test,
#     books_config=books_config,
#     anti_dict_file=anti_dict_file
# )

# %%
# =============================================================================
# VERIFY SUBMISSION
# =============================================================================

print("\n" + "="*60)
print("SUBMISSION VERIFICATION")
print("="*60)

print("\n📊 Submission Overview:")
print(df_submission.head(10))

print("\n📈 Statistics:")
print(f"Total entries: {len(df_submission)}")
print(f"Expected: 37 chapters (19 paf + 18 lca)")

# Check for missing chapters
expected_ids = [f"paf{i}" for i in range(19)] + [f"lca{i}" for i in range(18)]
actual_ids = set(df_submission.index)
missing_ids = [id for id in expected_ids if id not in actual_ids]

if missing_ids:
    print(f"\n⚠️  Missing chapters: {', '.join(missing_ids)}")
else:
    print("\n✅ All expected chapters present!")

# %%
# =============================================================================
# INSPECT SAMPLE GRAPHS
# =============================================================================

print("\n" + "="*60)
print("SAMPLE GRAPH INSPECTION")
print("="*60)

sample_ids = ["paf0", "lca0"]

for sample_id in sample_ids:
    if sample_id in df_submission.index:
        print(f"\n📖 Sample: {sample_id}")

        # Parse GraphML back to NetworkX
        import io
        graphml_str = df_submission.loc[sample_id, "graphml"]
        G_sample = nx.read_graphml(io.StringIO(graphml_str))

        print(f"   Nodes: {G_sample.number_of_nodes()}")
        print(f"   Edges: {G_sample.number_of_edges()}")

        print("\n   Sample nodes with 'names' attribute:")
        for i, (node, data) in enumerate(G_sample.nodes(data=True)):
            if i < 5:
                names = data.get('names', '⚠️ MISSING')
                print(f"      • {node}: {names}")
            if i == 5:
                print(f"      ... and {G_sample.number_of_nodes() - 5} more")
                break

In [ ]:
# %%
# =============================================================================
# CREATE INTERACTIVE HTML VISUALIZATIONS
# =============================================================================

print("\n" + "="*60)
print("INTERACTIVE HTML VISUALIZATION")
print("="*60)

# Select chapters to visualize (you can change these)
chapters_to_visualize = ["paf0", "lca0"]

for chapter_id in chapters_to_visualize:
    if chapter_id in df_submission.index:
        print(f"\n🎨 Creating interactive visualization for: {chapter_id}")
        
        # Parse GraphML back to NetworkX
        import io
        from collections import Counter
        graphml_str = df_submission.loc[chapter_id, "graphml"]
        G = nx.read_graphml(io.StringIO(graphml_str))
        
        # Reconstruct cooccurrences from edge weights
        cooccurrences = Counter()
        for u, v, data in G.edges(data=True):
            weight = int(float(data.get('weight', 1)))
            cooccurrences[(u, v)] = weight
        
        # Create interactive HTML file
        output_file = f"{chapter_id}_network.html"
        create_interactive_graph(G, cooccurrences, output_file=output_file)
        
        print(f"   ✓ Saved to: {output_file}")
        print(f"   📊 {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    else:
        print(f"\n⚠️  Chapter {chapter_id} not found in submission")

print("\n" + "="*60)
print("✅ Visualization complete! Open the HTML files to view interactive graphs.")
print("="*60)

In [ ]:
# %%
# =============================================================================
# MANUAL CHARACTER REVIEW - Identify False Positives
# =============================================================================
# This cell allows you to inspect extracted characters from any chapter
# Run this AFTER the main pipeline to review and identify false positives

import pandas as pd
from collections import Counter

def inspect_chapter_characters(chapter_id, books_folder, anti_dict_file, display_limit=None):
    """
    Extract and display all characters from a chapter for manual review.
    
    Args:
        chapter_id: e.g., "paf0", "lca5"
        books_folder: Path to data folder
        anti_dict_file: Path to antidict.txt
        display_limit: Max characters to show (None = all)
    """
    # Parse chapter_id
    if chapter_id.startswith("paf"):
        book_code = "paf"
        folder_name = "prelude_a_fondation"
        chapter_num = int(chapter_id[3:])
    elif chapter_id.startswith("lca"):
        book_code = "lca"
        folder_name = "les_cavernes_d_acier"
        chapter_num = int(chapter_id[3:])
    else:
        print(f"❌ Invalid chapter_id: {chapter_id}")
        return
    
    # Build file path
    chapter_file = f"{books_folder}/{folder_name}/chapter_{chapter_num + 1}.txt.preprocessed"
    
    if not os.path.exists(chapter_file):
        print(f"❌ File not found: {chapter_file}")
        return
    
    print(f"\n{'='*70}")
    print(f"📖 ANALYZING CHAPTER: {chapter_id.upper()}")
    print(f"{'='*70}")
    print(f"File: {chapter_file}")
    print()
    
    # Read text
    text = read_file(chapter_file)
    print(f"📝 Text length: {len(text):,} characters")
    
    # Load anti-dictionary
    anti_dict = load_anti_dict(anti_dict_file)
    print(f"🚫 Anti-dictionary: {len(anti_dict)} entries")
    print()
    
    # Step 1: Extract ALL entities
    print("="*70)
    print("STEP 1: Raw Entity Extraction (NER)")
    print("="*70)
    raw_entities = ensemble_entities(text, method="vote")
    L = count_entities(raw_entities)
    
    print(f"✓ Extracted {len(L)} unique entities (all types)")
    print()
    
    # Show all entity types
    entity_types = Counter([label for (text, label), count in L.items()])
    print("Entity Type Distribution:")
    for label, count in entity_types.most_common():
        print(f"  • {label:10} : {count:3} entities")
    print()
    
    # Step 2: Filter PERSONS only
    print("="*70)
    print("STEP 2: Person Entities (LP) - After Filtering")
    print("="*70)
    LP = filter_persons(L, anti_dict=anti_dict)
    
    print(f"✓ Found {len(LP)} person entities (after anti-dict filtering)")
    print()
    
    # Display persons with counts
    print("📋 All Person Entities (sorted by frequency):")
    print("-"*70)
    
    persons_list = []
    for i, (name, count) in enumerate(LP.most_common(display_limit), 1):
        persons_list.append({
            '#': i,
            'Name': name,
            'Count': count,
            'False Positive?': ''  # For manual marking
        })
        print(f"{i:3}. {name:40} | Count: {count:4}")
    
    print("-"*70)
    print()
    
    # Step 3: Show Alias Groups
    print("="*70)
    print("STEP 3: Alias Grouping")
    print("="*70)
    groups = group_aliases(LP)
    alias_map = alias_dictionary(groups)
    LP_merged = merge_alias_counts(LP, alias_map)
    
    print(f"✓ Grouped into {len(groups)} unique characters")
    print()
    
    print("📋 Alias Groups (merged characters):")
    print("-"*70)
    
    for i, group in enumerate(groups, 1):
        canonical = group[0]
        count = LP_merged[canonical]
        aliases = ", ".join(group[1:]) if len(group) > 1 else "(no aliases)"
        print(f"{i:3}. {canonical:30} | Count: {count:4}")
        if len(group) > 1:
            print(f"     Aliases: {aliases}")
    
    print("-"*70)
    print()
    
    # Step 4: Show potential false positives
    print("="*70)
    print("STEP 4: Potential False Positives (Low Frequency)")
    print("="*70)
    print("Characters mentioned only 1-2 times (often false positives):")
    print()
    
    low_freq = {name: count for name, count in LP_merged.items() if count <= 2}
    
    if low_freq:
        for i, (name, count) in enumerate(sorted(low_freq.items(), key=lambda x: x[1], reverse=True), 1):
            print(f"  {i:2}. {name:40} | Count: {count}")
    else:
        print("  ✓ No low-frequency characters found")
    
    print()
    
    # Summary
    print("="*70)
    print("📊 SUMMARY")
    print("="*70)
    print(f"Total entities (all types):       {len(L)}")
    print(f"Person entities (raw):            {len(LP)}")
    print(f"Unique characters (after merge):  {len(LP_merged)}")
    print(f"Low frequency (≤2 mentions):      {len(low_freq)}")
    print()
    
    # Return data for further analysis
    return {
        'chapter_id': chapter_id,
        'all_entities': L,
        'persons_raw': LP,
        'persons_merged': LP_merged,
        'alias_groups': groups,
        'low_frequency': low_freq,
        'persons_df': pd.DataFrame(persons_list)
    }


# =============================================================================
# USAGE: Inspect a specific chapter
# =============================================================================

# Change this to the chapter you want to analyze
CHAPTER_TO_INSPECT = "paf0"  # Options: paf0-paf18, lca0-lca17

# Run the inspection
results = inspect_chapter_characters(
    chapter_id=CHAPTER_TO_INSPECT,
    books_folder=books_folder,
    anti_dict_file=anti_dict_file,
    display_limit=None  # Show all characters (or set to 50 for top 50)
)

# Display as DataFrame for easier viewing
if results:
    print("\n" + "="*70)
    print("📊 DATAFRAME VIEW (for easier review)")
    print("="*70)
    display(results['persons_df'])
    
    # Export to file for manual editing
    export_file = f"{CHAPTER_TO_INSPECT}_characters.csv"
    results['persons_df'].to_csv(export_file, index=False)
    print(f"\n💾 Exported to: {export_file}")
    print("   You can open this in Excel/Sheets to mark false positives!")

In [ ]:
# %%
# =============================================================================
# COMPARE MULTIPLE CHAPTERS - Find Common False Positives
# =============================================================================
# Use this to find entities that appear across multiple chapters
# (likely real characters vs. false positives)

def compare_chapters(chapter_ids, books_folder, anti_dict_file):
    """
    Compare character extraction across multiple chapters.
    Helps identify common patterns and false positives.
    """
    print(f"\n{'='*70}")
    print(f"🔍 COMPARING {len(chapter_ids)} CHAPTERS")
    print(f"{'='*70}")
    print()
    
    all_characters = {}
    
    for chapter_id in chapter_ids:
        # Parse chapter_id
        if chapter_id.startswith("paf"):
            folder_name = "prelude_a_fondation"
            chapter_num = int(chapter_id[3:])
        elif chapter_id.startswith("lca"):
            folder_name = "les_cavernes_d_acier"
            chapter_num = int(chapter_id[3:])
        else:
            continue
        
        chapter_file = f"{books_folder}/{folder_name}/chapter_{chapter_num + 1}.txt.preprocessed"
        
        if not os.path.exists(chapter_file):
            continue
        
        # Extract characters
        text = read_file(chapter_file)
        anti_dict = load_anti_dict(anti_dict_file)
        raw_entities = ensemble_entities(text, method="vote")
        L = count_entities(raw_entities)
        LP = filter_persons(L, anti_dict=anti_dict)
        
        # Group aliases
        groups = group_aliases(LP)
        alias_map = alias_dictionary(groups)
        LP_merged = merge_alias_counts(LP, alias_map)
        
        all_characters[chapter_id] = LP_merged
        print(f"✓ {chapter_id}: {len(LP_merged)} characters")
    
    print()
    
    # Find characters appearing in multiple chapters
    character_appearances = Counter()
    for chapter_id, chars in all_characters.items():
        for char in chars.keys():
            character_appearances[char] += 1
    
    print("="*70)
    print("📊 CHARACTER FREQUENCY ACROSS CHAPTERS")
    print("="*70)
    print()
    
    print("Characters in ALL chapters (likely main characters):")
    for char, count in character_appearances.most_common():
        if count == len(chapter_ids):
            total_mentions = sum(all_characters[ch].get(char, 0) for ch in chapter_ids)
            print(f"  ✓ {char:40} | {count}/{len(chapter_ids)} chapters, {total_mentions} total mentions")
    
    print()
    print("Characters in ONLY ONE chapter (likely false positives or minor characters):")
    singles = [(char, count) for char, count in character_appearances.items() if count == 1]
    for char, count in sorted(singles, key=lambda x: sum(all_characters[ch].get(x[0], 0) for ch in chapter_ids), reverse=True)[:20]:
        for chapter_id, chars in all_characters.items():
            if char in chars:
                mentions = chars[char]
                print(f"  ⚠️  {char:40} | {chapter_id}, {mentions} mentions")
    
    return character_appearances, all_characters


# USAGE: Compare first 3 chapters of each book
chapters_to_compare = ["paf0", "paf1", "paf2", "lca0", "lca1", "lca2"]

# UNCOMMENT TO RUN:
# char_freq, char_data = compare_chapters(
#     chapter_ids=chapters_to_compare,
#     books_folder=books_folder,
#     anti_dict_file=anti_dict_file
# )